# Tutorial 13-6: Autoregressive Generation & Hallucination
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. One Token at a Time
As discussed in lecture, a Transformer Decoder (like GPT-2 or GPT-3) generates a sequence $Pr(t_1, t_2, ... t_N)$ by multiplying the conditional probabilities of each token.

In this tutorial, we will use **GPT-2** to manually 'unroll' a sentence to see how the model's output at Step $N$ becomes its input for Step $N+1$.

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch.nn.functional as F

# Load GPT-2 (A Decoder-only model)
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors='pt')

print(f"Initial Prompt: {prompt}")
print(f"Initial Input IDs: {inputs['input_ids']}")

## 2. Manual Decoding Step
Let's perform one forward pass and look at the probabilities for the very next token. We will see that ' Paris' should have a very high probability.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    
    # Get the logits for the LAST token in the sequence
    next_token_logits = logits[0, -1, :]
    probs = F.softmax(next_token_logits, dim=-1)
    
    # Get top 5 candidates
    top_5_probs, top_5_indices = torch.topk(probs, 5)
    
    print("Top 5 candidates for the next token:")
    for i in range(5):
        token = tokenizer.decode([top_5_indices[i]])
        print(f"{i+1}. '{token}' with probability {top_5_probs[i]:.4f}")

## 3. Comparing Decoding Strategies
The way we pick from those probabilities matters:
* **Greedy Search:** Always pick the #1 candidate. This is fast but can lead to repetitive or 'boring' loops.
* **Top-k Sampling:** Randomly pick from the top $k$ candidates. This adds 'creativity' and variety.
* **Top-p (Nucleus) Sampling:** Pick from the smallest set of tokens whose cumulative probability exceeds $p$.

In [ ]:
# Greedy Generation
greedy_output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
print("Greedy Output:", tokenizer.decode(greedy_output[0]))

# Sampling Output (Creative)
sampling_output = model.generate(**inputs, max_new_tokens=10, do_sample=True, top_k=50, temperature=0.7)
print("Sampling Output:", tokenizer.decode(sampling_output[0]))

## 4. The 'Hallucination' Trap
Because the model is just predicting the next most likely word based on patterns, it has no internal 'truth' database. If we provide a misleading prompt, the model will confidently continue the pattern even if it's false.

In [ ]:
fake_prompt = "The first person to walk on the sun was a scientist named"
inputs_fake = tokenizer(fake_prompt, return_tensors='pt')

hallucination = model.generate(**inputs_fake, max_new_tokens=15, do_sample=True, top_p=0.95)
print("Prompt:", fake_prompt)
print("Hallucination:", tokenizer.decode(hallucination[0]))

## 5. Summary
* **Autoregressive Loop:** The model's own output is fed back in as input for the next step.
* **Probability-Driven:** Every word is a statistical guess.
* **Hallucinations:** These aren't 'bugs' in the traditional sense; they are a direct result of the model prioritizing linguistic plausibility over factual accuracy.
* **Decoding Strategy:** Choosing between Greedy and Sampling allows us to trade off between consistency and creativity.